# MTEB Integration and Submission Pipeline

The screening round is scored from a JSON file that MTEB produces. This notebook works out how to get a two-stage pipeline through MTEB's interface, then runs it over the full test split.

**The problem:** MTEB expects an encoder. It calls `encode()`, gets vectors back, and computes similarity itself. Execution reranking has to happen after similarity, and the interface offers no obvious hook there.

Three possible outcomes, best first:

1. MTEB supports reranking or lets the search step be overridden
2. The rerank can be smuggled into a custom encoder subclass
3. MTEB can't express it, and the results JSON has to be built by hand

### 1. Inspect the interface

Reading what MTEB actually calls, and what it expects in return.

In [ ]:
!pip install -q mteb sentence-transformers datasets

In [ ]:
import mteb, inspect
from mteb.models.abs_encoder import AbsEncoder

print("mteb version:", mteb.__version__)
print()
print(inspect.getsource(AbsEncoder))

mteb version: 2.21.0

class AbsEncoder(ABC):
    """Base class to indicate that this is a wrapper for a model.

    Also contains some utility functions for wrappers for working with prompts and instructions.

    Attributes:
        model: The model to be wrapped.
        mteb_model_meta: Metadata about the model.
        model_prompts: A dictionary of prompts to be used for encoding sentences.
        instruction_template: A template for formatting instructions. Can be a string with '{instruction}' or
            a callable that takes the instruction and prompt type and returns a formatted instruction.
        prompts_dict: A dictionary of prompts to be used for encoding sentences, overrides model_prompts if provided.
    """

    model: Any
    mteb_model_meta: ModelMeta | None = None
    model_prompts: dict[str, str] | None = None
    instruction_template: str | Callable[[str, PromptType | None], str] | None = None
    prompts_dict: dict[str, str] | None = None

    def get_prompt_

### 2. Finding the hook

`similarity()` is a regular overridable method, not abstract. MTEB calls it to turn two embedding sets into a score matrix, which is exactly where execution reranking needs to sit.

Two things to confirm:

- whether it receives the full corpus at once or in chunks
- how to get the raw texts into it, since it only sees embeddings

Reading the retrieval evaluator to find out.

In [ ]:
import mteb, pkgutil, inspect

print("top-level modules:")
for m in pkgutil.iter_modules(mteb.__path__):
    print("  ", m.name)

top-level modules:
   __main__
   _content_hashes
   _create_dataloaders
   _evaluators
   _helpful_enum
   _hf_integration
   _log_once
   _requires_package
   _reversible_workflow
   _set_seed
   abstasks
   api
   benchmarks
   cache
   cli
   data_cleaning
   deprecated_evaluator
   evaluate
   filter_tasks
   get_tasks
   languages
   leaderboard
   load_results
   mocks
   models
   results
   similarity_functions
   tasks
   timing
   types


In [ ]:
# find whatever class actually does the searching
import importlib, pkgutil

hits = []
for mod in pkgutil.walk_packages(mteb.__path__, prefix="mteb."):
    if any(w in mod.name.lower() for w in ["retriev", "search", "evaluat"]):
        hits.append(mod.name)
for h in hits:
    print(h)

/usr/local/lib/python3.13/dist-packages/mteb/abstasks/retrieval.py:110: UserWarning: The task 'AVMemeExamAT2VRetrieval' is currently in beta. This means that the dataset is still being tested and may be subject to changes. This means that the scores of this dataset is liable to change and should be used with caution.
  super().__init__(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/mteb/abstasks/retrieval.py:110: UserWarning: The task 'ActivityNetCaptionsT2VRetrieval' is currently in beta. This means that the dataset is still being tested and may be subject to changes. This means that the scores of this dataset is liable to change and should be used with caution.
  super().__init__(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/mteb/abstasks/retrieval.py:110: UserWarning: The task 'AudioCapsAVVA2TRetrieval' is currently in beta. This means that the dataset is still being tested and may be subject to changes. This means that the scores of this dataset is liable to ch

mteb._evaluators
mteb._evaluators.any_sts_evaluator
mteb._evaluators.classification_metrics
mteb._evaluators.clustering_evaluator
mteb._evaluators.evaluator
mteb._evaluators.image
mteb._evaluators.image.imagetext_pairclassification_evaluator
mteb._evaluators.pair_classification_evaluator
mteb._evaluators.retrieval_evaluator
mteb._evaluators.retrieval_metrics
mteb._evaluators.sklearn_evaluator
mteb._evaluators.text
mteb._evaluators.text.bitext_mining_evaluator
mteb._evaluators.text.summarization_evaluator
mteb._evaluators.zeroshot_classification_evaluator
mteb.abstasks.retrieval
mteb.abstasks.retrieval_dataset_loaders
mteb.data_cleaning._retrieval
mteb.deprecated_evaluator
mteb.evaluate
mteb.mocks.mock_tasks.retrieval
mteb.models.model_implementations.bmretriever_models
mteb.models.model_implementations.en_code_retriever
mteb.models.model_implementations.omniretriever_models
mteb.models.model_implementations.opensearch_neural_sparse_models
mteb.models.model_implementations.searchmap_mod

In [ ]:
import inspect
from mteb.models import search_wrappers

print([n for n in dir(search_wrappers) if not n.startswith('_')])
print()
src = inspect.getsource(search_wrappers)
print(src[:4000])

['Any', 'PromptType', 'SearchCrossEncoderWrapper', 'SearchEncoderWrapper', 'TYPE_CHECKING', 'annotations', 'create_dataloader', 'heapq', 'logger', 'logging']

from __future__ import annotations

import heapq
import logging
from typing import TYPE_CHECKING, Any

from mteb._create_dataloaders import (
    create_dataloader,
)
from mteb.types import (
    PromptType,
)

if TYPE_CHECKING:
    import torch
    from torch.utils.data import DataLoader

    from mteb.abstasks.task_metadata import TaskMetadata
    from mteb.types import (
        Array,
        BatchedInput,
        CorpusDatasetType,
        EncodeKwargs,
        QueryDatasetType,
        RetrievalOutputType,
        TopRankedDocumentsType,
    )

    from .models_protocols import CrossEncoderProtocol, EncoderProtocol
    from .search_encoder_index.search_backend_protocol import IndexEncoderSearchProtocol

logger = logging.getLogger(__name__)


class SearchEncoderWrapper:
    """Wrapper for Encoder models to be used in search 

### 3. The hook

`SearchEncoderWrapper.search()` returns a mapping of query IDs to document IDs and scores. Subclassing it means calling the parent for the embedding ranking, then reordering the top candidates by execution before returning.

Better than overriding `similarity()`, since this level has IDs rather than raw vectors, and `index()` already retains the corpus as `self.task_corpus`.

Checking the rest of `search()` for how queries arrive and how MTEB selects the wrapper.

In [ ]:
print(src[4000:9000])

        if self.task_corpus is None:
            raise ValueError("Corpus must be indexed before searching.")

        queries_dataloader = create_dataloader(
            queries,
            task_metadata=task_metadata,
            prompt_type=PromptType.query,
            num_proc=num_proc,
            **encode_kwargs,
        )

        query_embeddings = self.model.encode(
            queries_dataloader,
            task_metadata=task_metadata,
            hf_split=hf_split,
            hf_subset=hf_subset,
            prompt_type=PromptType.query,
            **encode_kwargs,
        )
        query_idx_to_id = dict(enumerate(queries["id"]))

        if top_ranked is not None:
            logger.info("Reranking pre-ranked documents...")
            if self.index_backend is None:
                result_heaps = self._rerank_documents(
                    query_idx_to_id=query_idx_to_id,
                    query_embeddings=query_embeddings,
                    top_ranked=top_ranked,

In [ ]:
# how does mteb decide to wrap an encoder in this?
import mteb.evaluate as ev
s = inspect.getsource(ev)
for i, l in enumerate(s.split("\n")):
    if "SearchEncoderWrapper" in l or "search_wrappers" in l:
        print(i, l)

### 4. The subclass design

`search()` has access to both sides: the `queries` dataset carries query text, and `self.task_corpus` holds the documents. The return value is a plain dict of query ID to document ID and score.

So the override is straightforward: call the parent to get the embedding ranking, then reorder the top candidates by execution.

One trap. `search()` sets `self.task_corpus = None` before returning, to free memory. A copy has to be kept during `index()`.

Still to confirm: whether MTEB wraps the encoder automatically or accepts a wrapper directly.

In [ ]:
import mteb.evaluate as ev, inspect
s = inspect.getsource(ev)
for i, l in enumerate(s.split("\n")):
    if any(w in l for w in ["SearchEncoderWrapper", "search_wrappers", "SearchProtocol", "isinstance(model"]):
        print(f"{i:4d} {l}")

 189     if isinstance(model, ModelMeta):


In [ ]:
# import pkgutil, importlib, inspect, mteb

# for mod in pkgutil.walk_packages(mteb.__path__, prefix="mteb."):
#     if any(x in mod.name for x in ["tasks.", "mocks", "leaderboard", "data_cleaning"]):
#         continue
#     try:
#         m = importlib.import_module(mod.name)
#         src = inspect.getsource(m)
#     except Exception:
#         continue
#     if "SearchEncoderWrapper(" in src:
#         print("==", mod.name)
#         for i, l in enumerate(src.split("\n")):
#             if "SearchEncoderWrapper" in l:
#                 print(f"   {i:4d} {l.strip()}")



!grep -rn "SearchEncoderWrapper(" /usr/local/lib/python3.13/dist-packages/mteb/ --include=*.py | grep -v "/tasks/"

/usr/local/lib/python3.13/dist-packages/mteb/models/hybrid_wrappers.py:88:                wrapped = SearchEncoderWrapper(model)
/usr/local/lib/python3.13/dist-packages/mteb/abstasks/retrieval.py:393:            search_model = SearchEncoderWrapper(model)


In [ ]:
!sed -n '370,410p' /usr/local/lib/python3.13/dist-packages/mteb/abstasks/retrieval.py

            Dictionary of evaluation scores
        """
        # ensure queries format (see #3030)
        data_split["relevant_docs"], data_split["queries"] = (
            _filter_queries_without_positives(
                data_split["relevant_docs"], data_split["queries"]
            )
        )
        retriever = RetrievalEvaluator(
            corpus=data_split["corpus"],
            queries=data_split["queries"],
            task_metadata=self.metadata,
            hf_split=hf_split,
            hf_subset=hf_subset,
            top_ranked=data_split["top_ranked"],
            top_k=self._top_k,
            timer=timer,
            **kwargs,
        )

        search_model: SearchProtocol

        if isinstance(model, EncoderProtocol) and not isinstance(model, SearchProtocol):
            search_model = SearchEncoderWrapper(model)
        elif isinstance(model, CrossEncoderProtocol):
            search_model = SearchCrossEncoderWrapper(model)
        elif isinstance(model, Searc

### 5. Passing the subclass directly

MTEB checks the model type and takes whatever already satisfies `SearchProtocol` without wrapping it. A subclass of `SearchEncoderWrapper` inherits `index()` and `search()`, so it qualifies and gets used as-is.

No monkey-patching needed. The override is just a subclass handed to `mteb.evaluate`.

Confirming what `SearchProtocol` requires before building it.

In [ ]:
import inspect
from mteb.models.models_protocols import SearchProtocol
print(inspect.getsource(SearchProtocol))

@runtime_checkable
class SearchProtocol(Protocol):
    """Interface for searching models."""

    def index(
        self,
        corpus: CorpusDatasetType,
        *,
        task_metadata: TaskMetadata,
        hf_split: str,
        hf_subset: str,
        encode_kwargs: EncodeKwargs,
        num_proc: int | None,
    ) -> None:
        """Index the corpus for retrieval.

        Args:
            corpus: Corpus dataset to index.
            task_metadata: Metadata of the task, used to determine how to index the corpus.
            hf_split: Split of current task, allows to know some additional information about current split.
            hf_subset: Subset of current task. Similar to `hf_split` to get more information
            encode_kwargs: Additional arguments to pass to the encoder during indexing.
            num_proc: Number of processes to use for dataloading.
        """
        ...

    def search(
        self,
        queries: QueryDatasetType,
        *,
        task_

### 6. The encoder

The inner model MTEB calls for embeddings. E5 prefixes are applied here, using `prompt_type` to tell queries from documents.

Inputs arrive as a DataLoader of batches rather than a plain list, so the text has to be pulled out of each batch first.

In [ ]:
import numpy as np, torch
from sentence_transformers import SentenceTransformer
from mteb.models.abs_encoder import AbsEncoder
from mteb.models.model_meta import ModelMeta
from mteb.types import PromptType

class E5Encoder(AbsEncoder):
    def __init__(self, name="intfloat/e5-base-v2"):
        self.model = SentenceTransformer(name, device="cuda" if torch.cuda.is_available() else "cpu")
        self.model.max_seq_length = 512
        self.mteb_model_meta = None

    def encode(self, inputs, *, task_metadata, hf_split, hf_subset,
               prompt_type=None, **kwargs):
        prefix = "query: " if prompt_type == PromptType.query else "passage: "
        texts = []
        for batch in inputs:
            texts.extend(batch["text"])
        return self.model.encode([prefix + t for t in texts],
                                 batch_size=kwargs.get("batch_size", 64),
                                 normalize_embeddings=True,
                                 convert_to_numpy=True,
                                 show_progress_bar=True)

In [ ]:
# smoke test: does mteb accept it at all?
import mteb
task = mteb.get_task("AppsRetrieval")
print(task.metadata.name, task.metadata.eval_splits)

enc = E5Encoder()
print("encoder built ok")

AppsRetrieval ['test']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoder built ok


### 7. Baseline through MTEB

Running the plain encoder end to end first. No reranking yet.

Two things this confirms: that MTEB accepts the encoder and that the batch key is right, and that the score lands near the 13.11 measured by hand in notebook 01. A large gap would mean something is wired wrong.

In [ ]:
import time
t0 = time.time()
result = mteb.evaluate(E5Encoder(), [task], encode_kwargs={"batch_size": 64})
print(f"\n{time.time()-t0:.0f}s")

tr = list(result.task_results)[0]
d = tr.to_dict()
scores = d["scores"]["test"][0]
print("NDCG@10:", round(scores["ndcg_at_10"]*100, 2))
print("MRR@10 :", round(scores["mrr_at_10"]*100, 2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 57.5kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.61MB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating queries split:   0%|          | 0/8765 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8765 [00:00<?, ? examples/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

### 8. Adding model metadata

The evaluation ran but failed while caching, since `mteb_model_meta` was `None` and the cache builds its folder path from the model name.

Supplying a `ModelMeta` fixes it. The name also becomes the identifier in the results JSON, so it should describe the actual pipeline rather than the underlying model.

In [ ]:
from mteb.models.model_meta import ModelMeta

META = ModelMeta(
    name="prism/e5-base-v2-baseline",
    revision="1",
    release_date="2026-09-17",
    languages=["eng-Latn"],
    loader=None,
    memory_usage_mb=418,
    n_parameters=110_000_000,
    max_tokens=512,
    embed_dim=768,
    license="mit",
    open_weights=True,
    public_training_code=None,
    public_training_data=None,
    framework=["Sentence Transformers"],
    similarity_fn_name="cosine",
    use_instructions=True,
    training_datasets=None,
)

class E5Encoder(AbsEncoder):
    def __init__(self, name="intfloat/e5-base-v2"):
        self.model = SentenceTransformer(name, device="cuda" if torch.cuda.is_available() else "cpu")
        self.model.max_seq_length = 512
        self.mteb_model_meta = META

    def encode(self, inputs, *, task_metadata, hf_split, hf_subset,
               prompt_type=None, **kwargs):
        prefix = "query: " if prompt_type == PromptType.query else "passage: "
        texts = []
        for batch in inputs:
            texts.extend(batch["text"])
        return self.model.encode([prefix + t for t in texts],
                                 batch_size=kwargs.get("batch_size", 64),
                                 normalize_embeddings=True,
                                 convert_to_numpy=True,
                                 show_progress_bar=True)

print("ok:", META.name)

ok: prism/e5-base-v2-baseline


### 9. Baseline confirmed

11.52 NDCG@10, matching the CoIR paper's reported figure for E5-base on APPS exactly.

The difference from notebook 01's 13.11 is corpus size. MTEB indexes all 8,765 documents including the train partition, while notebook 01 searched only the 3,765 test documents. Larger corpus, lower score.

Total runtime 264 seconds on a T4.

### 10. The reranking subclass

Subclassing `SearchEncoderWrapper` to insert execution reranking after the embedding search.

Three changes to the parent:

- `index()` keeps its own copy of the corpus, since the parent clears `task_corpus` at the end of `search()`
- `search()` calls the parent, then reorders each query's candidates by whether they produce the expected output
- scores are rewritten so passing snippets sort above the rest, keeping embedding order within each group

In [ ]:
import re, subprocess, tempfile, os
from mteb.models.search_wrappers import SearchEncoderWrapper

EX = re.compile(r'-----Examples?-----\s*Input\s*\n(.*?)\n\s*Output\s*\n(.*?)(?:\n\s*Input\s*\n|\n-----|\Z)', re.S)

def run_snippet(code, stdin_data, timeout=1):
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(code); path = f.name
    try:
        r = subprocess.run(['python3', path], input=stdin_data,
                           capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip()
    except Exception:
        return None
    finally:
        os.unlink(path)

class ExecutionReranker(SearchEncoderWrapper):
    def __init__(self, model, K=50, timeout=1):
        super().__init__(model)
        self.K, self.timeout = K, timeout
        self.doc_text = {}

    def index(self, corpus, **kw):
        self.doc_text = dict(zip(corpus["id"], corpus["text"]))
        return super().index(corpus, **kw)

    def search(self, queries, **kw):
        results = super().search(queries, **kw)
        qtext = dict(zip(queries["id"], queries["text"]))

        for qid, hits in results.items():
            m = EX.search(qtext[qid])
            if not m: continue
            inp, exp = m.group(1).strip(), m.group(2).strip()

            ranked = sorted(hits.items(), key=lambda x: -x[1])[:self.K]
            passers = [d for d, _ in ranked
                       if run_snippet(self.doc_text[d], inp + "\n", self.timeout) == exp]
            if not passers: continue

            boost = max(hits.values()) + 1.0
            for d in passers:
                hits[d] = boost + hits[d]
        return results

print("reranker defined")

### 11. Smoke test on a subset

The full run is hours, so the plumbing gets checked on a handful of queries first.

Bypassing `mteb.evaluate` and calling `index()` and `search()` directly, which makes it obvious whether the reranker fires and whether scores actually move.

In [ ]:
print(type(task.dataset))
print(list(task.dataset.keys()))

In [ ]:
k = list(task.dataset.keys())[0]
print("first key:", k)
print(type(task.dataset[k]))
print(list(task.dataset[k].keys()) if hasattr(task.dataset[k], 'keys') else task.dataset[k])

In [ ]:
ds = task.dataset["default"]["test"]
print(type(ds))
print(list(ds.keys()) if hasattr(ds, 'keys') else ds)

In [ ]:
ds = task.dataset["default"]["test"]
corpus_full, queries_full, qrels = ds["corpus"], ds["queries"], ds["relevant_docs"]
print("corpus", len(corpus_full), "| queries", len(queries_full))
print("corpus cols:", corpus_full.column_names if hasattr(corpus_full,'column_names') else type(corpus_full))
print("query cols :", queries_full.column_names if hasattr(queries_full,'column_names') else type(queries_full))

In [ ]:
import numpy as np

N = 20
q_small = queries_full.select(range(N))
qids = list(q_small["id"])
gold = {q: list(qrels[q].keys())[0] for q in qids}

def rank_of_gold(results):
    out = {}
    for q in qids:
        ranked = sorted(results[q].items(), key=lambda x: -x[1])
        ids = [d for d, _ in ranked]
        out[q] = ids.index(gold[q]) + 1 if gold[q] in ids else 999
    return out

kw = dict(task_metadata=task.metadata, hf_split="test", hf_subset="default",
          encode_kwargs={"batch_size": 64}, num_proc=None)

# baseline
base = ExecutionReranker(E5Encoder(), K=0)
base.index(corpus_full, **kw)
r_base = base.search(q_small, top_k=1000, **kw)

# with rerank
rr = ExecutionReranker(E5Encoder(), K=50)
rr.index(corpus_full, **kw)
r_rr = rr.search(q_small, top_k=1000, **kw)

b, a = rank_of_gold(r_base), rank_of_gold(r_rr)
for q in qids:
    flag = "" if b[q] == a[q] else ("  <-- improved" if a[q] < b[q] else "  <-- HURT")
    print(f"{q}: {b[q]:4d} -> {a[q]:4d}{flag}")

### 12. Smoke test result

Two improvements, no regressions, on 20 queries. The reranker fires correctly through MTEB's interface.

Only candidates inside the top 50 are eligible, so most queries here were never in scope. Of the five with gold ranked at or under 50, two were already near the top and two jumped to rank 1.

Note the corpus gets re-encoded on every call, which costs two minutes each time. Caching the document embeddings would make iteration much faster.

### 13. Making the run survivable

The full evaluation runs for hours, and Colab disconnects idle sessions. Without checkpoints a failure at any point loses everything.

Three changes:

- document embeddings cached to Drive, so a restart skips the two-minute encode
- execution results written per query as they complete
- the run skips any query already present in the checkpoint file

Drive rather than local disk, since the Colab filesystem is wiped on disconnect.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

# import os
# print(os.path.exists('/content/drive/MyDrive/prism/exec_results.jsonl'))

# import os
# print(os.path.exists('/content/drive'))
# print(os.listdir('/content/drive') if os.path.exists('/content/drive') else 'no /content/drive')
# print(os.listdir('/content/drive/MyDrive')[:20] if os.path.exists('/content/drive/MyDrive') else 'no MyDrive')

# import os
# print(os.listdir('/content/drive/MyDrive/prism'))

# !ls -la /content/drive/MyDrive/ | head -20

import os
WORK = '/content/drive/MyDrive/prism'
p = f'{WORK}/exec_results.jsonl'
print(os.path.exists(p), sum(1 for _ in open(p)), "lines")

True 3765 lines


### 14. Caching document embeddings

Encoding all 8,765 documents takes two minutes. Saving the vectors to Drive means a restart skips it.

This is also the index-build cost that P1 is judged on, so the timing is worth recording.

In [ ]:
import os, numpy as np, time

WORK = '/content/drive/MyDrive/prism'
os.makedirs(WORK, exist_ok=True)
EMB = f'{WORK}/doc_emb.npy'
IDS = f'{WORK}/doc_ids.npy'

ds = task.dataset["default"]["test"]
corpus_full, queries_full, qrels = ds["corpus"], ds["queries"], ds["relevant_docs"]

if os.path.exists(EMB):
    doc_emb = np.load(EMB)
    doc_ids = np.load(IDS, allow_pickle=True)
    print("loaded from cache:", doc_emb.shape)
else:
    enc = E5Encoder()
    t0 = time.time()
    doc_emb = enc.model.encode(["passage: " + t for t in corpus_full["text"]],
                               batch_size=64, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=True)
    doc_ids = np.array(corpus_full["id"])
    np.save(EMB, doc_emb); np.save(IDS, doc_ids)
    print(f"encoded in {time.time()-t0:.0f}s, shape {doc_emb.shape}")

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!cat /proc/cpuinfo | grep -c processor

### 15. Precomputing the baseline ranking

Execution reranking only touches the top K, so the embedding ranking can be computed once for all queries and reused.

Encoding queries, scoring against the cached document embeddings, and saving the top 200 candidates per query. That file then feeds the execution pass, which becomes a pure CPU job with no model in the loop.

In [ ]:
task.load_data()
ds = task.dataset["default"]["test"]
corpus_full, queries_full, qrels = ds["corpus"], ds["queries"], ds["relevant_docs"]
print(len(corpus_full), len(queries_full), len(qrels))

8765 3765 3765


In [ ]:
import numpy as np, time, os

RANK = f'{WORK}/base_ranking.npz'

if os.path.exists(RANK):
    z = np.load(RANK, allow_pickle=True)
    top_idx, top_score, q_ids = z['idx'], z['score'], z['qids']
    print("loaded:", top_idx.shape)
else:
    enc = E5Encoder()
    t0 = time.time()
    q_emb = enc.model.encode(["query: " + t for t in queries_full["text"]],
                             batch_size=64, normalize_embeddings=True,
                             convert_to_numpy=True, show_progress_bar=True)
    sim = q_emb @ doc_emb.T
    top_idx = np.argsort(-sim, axis=1)[:, :200]
    top_score = np.take_along_axis(sim, top_idx, axis=1)
    q_ids = np.array(queries_full["id"])
    np.savez(RANK, idx=top_idx, score=top_score, qids=q_ids)
    print(f"done in {time.time()-t0:.0f}s, shape {top_idx.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 16. The execution pass

The long one. For each query, run the top K candidates against the example input and record which produce the expected output.

Results append to a JSONL file on Drive as each query completes, and the run skips anything already recorded. So a disconnect costs only the query in flight.

No model involved from here. Two cores, roughly 60ms per candidate.

In [ ]:
import json, re, subprocess, tempfile, os, time
from concurrent.futures import ThreadPoolExecutor

OUT = f'{WORK}/exec_results.jsonl'
K = 50
TIMEOUT = 1

doc_text = list(corpus_full["text"])
q_text   = list(queries_full["text"])
EX = re.compile(r'-----Examples?-----\s*Input\s*\n(.*?)\n\s*Output\s*\n(.*?)(?:\n\s*Input\s*\n|\n-----|\Z)', re.S)

def run_snippet(code, stdin_data, timeout=TIMEOUT):
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(code); path = f.name
    try:
        r = subprocess.run(['python3', path], input=stdin_data,
                           capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip()
    except Exception:
        return None
    finally:
        try: os.unlink(path)
        except: pass

done = set()
if os.path.exists(OUT):
    with open(OUT) as f:
        for line in f:
            try: done.add(json.loads(line)['qid'])
            except: pass
print(f"already done: {len(done)}")

t_start = time.time()
with open(OUT, 'a') as fout:
    for i, qid in enumerate(q_ids):
        if qid in done: continue
        m = EX.search(q_text[i])
        if not m:
            fout.write(json.dumps({"qid": str(qid), "passers": []}) + "\n"); fout.flush()
            continue
        inp, exp = m.group(1).strip(), m.group(2).strip()
        cand = top_idx[i][:K]
        with ThreadPoolExecutor(max_workers=2) as ex:
            outs = list(ex.map(lambda j: run_snippet(doc_text[j], inp + "\n"), cand))
        passers = [int(cand[n]) for n, o in enumerate(outs) if o == exp]
        fout.write(json.dumps({"qid": str(qid), "passers": passers}) + "\n"); fout.flush()

        if (i+1) % 50 == 0:
            el = time.time() - t_start
            rate = (i+1-len(done)) / el if el > 0 else 0
            print(f"{i+1}/{len(q_ids)}  {el/60:.0f}min elapsed  ~{(len(q_ids)-i-1)/rate/60:.0f}min left")

print("complete")

### 17. The result

Applying the execution results to the baseline ranking. Snippets that produced the expected output move above the rest, keeping their embedding order within each group.

Computing NDCG@10 and MRR on the full test split, against the 11.52 baseline.

In [ ]:
import os, json, numpy as np, mteb

WORK = '/content/drive/MyDrive/prism'
OUT  = f'{WORK}/exec_results.jsonl'
RANK = f'{WORK}/base_ranking.npz'
IDS  = f'{WORK}/doc_ids.npy'

z = np.load(RANK, allow_pickle=True)
top_idx, top_score, q_ids = z['idx'], z['score'], z['qids']
doc_ids = np.load(IDS, allow_pickle=True)

task = mteb.get_task("AppsRetrieval")
task.load_data()
ds = task.dataset["default"]["test"]
corpus_full, queries_full, qrels = ds["corpus"], ds["queries"], ds["relevant_docs"]

print("ranking:", top_idx.shape, "| docs:", len(doc_ids))
print("exec lines:", sum(1 for _ in open(OUT)))

In [1]:
!nvidia-smi --query-gpu=name --format=csv,noheader

Tesla T4


In [2]:
!pip install -q mteb sentence-transformers datasets
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/prism'
print(os.listdir(WORK))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 28.4 MB/s eta 0:00:00
Mounted at /content/drive
['doc_emb.npy', 'doc_ids.npy', 'base_ranking.npz', 'exec_results.jsonl']


In [3]:
import mteb
task = mteb.get_task("AppsRetrieval")
task.load_data()
ds = task.dataset["default"]["test"]
corpus_full, queries_full, qrels = ds["corpus"], ds["queries"], ds["relevant_docs"]
print(len(corpus_full), len(queries_full), len(qrels))

README.md:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 57.5kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.7kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.61MB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating queries split:   0%|          | 0/8765 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8765 [00:00<?, ? examples/s]

8765 3765 3765


In [5]:
import numpy as np
z = np.load(f'{WORK}/base_ranking.npz', allow_pickle=True)
top_idx, q_ids = z['idx'], z['qids']
doc_ids = np.load(f'{WORK}/doc_ids.npy', allow_pickle=True)
print(top_idx.shape, len(doc_ids))

(3765, 200) 8765


In [6]:
import numpy as np, time
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/e5-base-v2", device="cuda")
model.max_seq_length = 512

t0 = time.time()
doc_emb = model.encode(["passage: " + t for t in corpus_full["text"]], batch_size=64,
                       normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
q_emb = model.encode(["query: " + t for t in queries_full["text"]], batch_size=64,
                     normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
print(f"encoded in {time.time()-t0:.0f}s")

sim = q_emb @ doc_emb.T
top_idx = np.argsort(-sim, axis=1)[:, :200]
doc_ids = np.array(corpus_full["id"])
q_ids   = np.array(queries_full["id"])
np.savez(f'{WORK}/base_ranking.npz', idx=top_idx, qids=q_ids)
np.save(f'{WORK}/doc_ids.npy', doc_ids)
print("saved")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/137 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

encoded in 248s
saved


In [7]:
import json, numpy as np

exec_res = {}
for line in open(f'{WORK}/exec_results.jsonl'):
    d = json.loads(line); exec_res[d['qid']] = set(d['passers'])

id_to_pos = {d: i for i, d in enumerate(doc_ids)}
old_r, new_r = [], []
for i, qid in enumerate(q_ids):
    g = id_to_pos[list(qrels[qid].keys())[0]]
    cand = list(top_idx[i])
    old = cand.index(g) + 1 if g in cand else 999
    p = exec_res.get(str(qid), set())
    order = [c for c in cand if c in p] + [c for c in cand if c not in p] if p else cand
    new = order.index(g) + 1 if g in order else 999
    old_r.append(old); new_r.append(new)

old_r, new_r = np.array(old_r), np.array(new_r)
ndcg = lambda r: np.where(r <= 10, 1/np.log2(r+1), 0).mean()*100
mrr  = lambda r: np.where(r < 999, 1/r, 0).mean()*100
print(f"NDCG@10  {ndcg(old_r):.2f} -> {ndcg(new_r):.2f}")
print(f"MRR      {mrr(old_r):.2f} -> {mrr(new_r):.2f}")
print(f"improved {(new_r<old_r).sum()}, hurt {(new_r>old_r).sum()}, same {(new_r==old_r).sum()}")

NDCG@10  11.52 -> 19.96
MRR      10.52 -> 19.43
improved 420, hurt 10, same 3335


### 18. Generating the submission JSON

MTEB has to produce the submitted file, so the reranker goes back through `mteb.evaluate`.

Execution results are already on disk from the full pass. This subclass loads them instead of running any programs, then boosts passing snippets above the rest of each query's results.

The score MTEB reports should match the 19.96 computed by hand.

In [8]:
import torch, json
from sentence_transformers import SentenceTransformer
from mteb.models.abs_encoder import AbsEncoder
from mteb.models.model_meta import ModelMeta
from mteb.models.search_wrappers import SearchEncoderWrapper
from mteb.types import PromptType

META = ModelMeta(
    name="prism/e5-base-v2-exec-rerank", revision="1", release_date="2026-09-24",
    languages=["eng-Latn"], loader=None, memory_usage_mb=418,
    n_parameters=110_000_000, max_tokens=512, embed_dim=768, license="mit",
    open_weights=True, public_training_code=None, public_training_data=None,
    framework=["Sentence Transformers"], similarity_fn_name="cosine",
    use_instructions=True, training_datasets=None,
)

class E5Encoder(AbsEncoder):
    def __init__(self, name="intfloat/e5-base-v2"):
        self.model = SentenceTransformer(name, device="cuda" if torch.cuda.is_available() else "cpu")
        self.model.max_seq_length = 512
        self.mteb_model_meta = META

    def encode(self, inputs, *, task_metadata, hf_split, hf_subset, prompt_type=None, **kwargs):
        prefix = "query: " if prompt_type == PromptType.query else "passage: "
        texts = [t for batch in inputs for t in batch["text"]]
        return self.model.encode([prefix + t for t in texts], batch_size=64,
                                 normalize_embeddings=True, convert_to_numpy=True,
                                 show_progress_bar=True)

class CachedExecReranker(SearchEncoderWrapper):
    def __init__(self, model, passers_by_qid):
        super().__init__(model)
        self.passers = passers_by_qid

    def search(self, queries, **kw):
        results = super().search(queries, **kw)
        for qid, hits in results.items():
            p = self.passers.get(qid)
            if not p or not hits:
                continue
            boost = max(hits.values()) + 1.0
            for d in p:
                if d in hits:
                    hits[d] = boost + hits[d]
        return results

# corpus positions -> document ids
passers_by_qid = {}
for line in open(f'{WORK}/exec_results.jsonl'):
    d = json.loads(line)
    if d['passers']:
        passers_by_qid[d['qid']] = {doc_ids[i] for i in d['passers']}
print("queries with passers:", len(passers_by_qid))

queries with passers: 924


In [9]:
import mteb, time

t0 = time.time()
result = mteb.evaluate(CachedExecReranker(E5Encoder(), passers_by_qid),
                       [mteb.get_task("AppsRetrieval")],
                       encode_kwargs={"batch_size": 64})
print(f"{time.time()-t0:.0f}s")

task_result = list(result.task_results)[0]
out = task_result.to_dict()
s = out["scores"]["test"][0]
print("NDCG@10:", round(s["ndcg_at_10"]*100, 2))
print("MRR@10 :", round(s["mrr_at_10"]*100, 2))

with open("appsretrieval_results.json", "w") as f:
    json.dump(out, f, indent=2)
with open(f"{WORK}/appsretrieval_results.json", "w") as f:
    json.dump(out, f, indent=2)
print("saved")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

Standardizing text corpus format:   0%|          | 0/8765 [00:00<?, ? examples/s]

Batches:   0%|          | 0/137 [00:00<?, ?it/s]

270s
NDCG@10: 19.96
MRR@10 : 19.05


TypeError: Object of type datetime is not JSON serializable

In [10]:
for path in ["appsretrieval_results.json", f"{WORK}/appsretrieval_results.json"]:
    with open(path, "w") as f:
        json.dump(out, f, indent=2, default=str)
print("saved")

saved
